# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, tell you honestly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [62]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=15)
            if r.status_code == 200:
                return r.json()
            if r.status_code not in RETRY_STATUSES:
                raise RuntimeError(
                    f'will not retry: returned {r.status_code} {r.text[:200]}')
            last = f'returned {r.status_code}'
        except requests.RequestException as e:
            last = repr(e)
        delay = 0.5 * (2 ** attempt)
        print(f"retrying in {delay}")
        time.sleep(delay)
    raise RuntimeError(f'gave up after {tries} attempts: {last}')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-10**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [63]:
from datetime import datetime

# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False

start = '2026-09-05'
end = '2026-10-10'
now = datetime.now().strftime('%Y-%m-%d %H:%M')
data = get_weather('2026-09-05', '2026-10-10')

wx = pd.DataFrame(data['daily'])
wx = pd.concat([wx.head(1), wx.tail(1)])
wx = wx.reset_index(drop=True)

wx

,time,temperature_2m_max,precipitation_sum
0,2026-09-05,29.4,2.5
1,2026-10-10,19.2,13.1


### Part 3 — the fallback, done honestly

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [64]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-10'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

wx['source'] = 'open-meteo'
wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-10,19.2,13.1,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [65]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['date'])
wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-10,19.2,13.1,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [66]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [67]:
wx_sorted = wx.sort_values(by='precip_mm', ascending=False)
print(f"{wx_sorted.iloc[0]['date'].strftime('%Y-%m-%d')} was wetter by {wx_sorted.iloc[0]['precip_mm'] - wx_sorted.iloc[1]['precip_mm']}mm")

2026-10-10 was wetter by 10.6mm


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [68]:
print('Source:  Open-Meteo historical archive API')
print('Endpoint:', BASE)
print('Date Range: ', start, end)
print('Pulled: ', now)

Source:  Open-Meteo historical archive API
Endpoint: https://archive-api.open-meteo.com/v1/archive
Date Range:  2026-09-05 2026-10-10
Pulled:  2026-10-09 23:40


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

_The chart would show values of nothing or a meaningless trend. No one would catch it because nothing would crash._

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [69]:
data_orlando = get_weather('2026-09-05', '2026-10-10', lat=28.54, lon=-81.38)

wx_orlando = pd.DataFrame(data_orlando['daily'])
wx_orlando = pd.concat([wx_orlando.head(1), wx_orlando.tail(1)]).reset_index(drop=True)

wx_orlando = wx_orlando.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})
wx_orlando['date'] = pd.to_datetime(wx_orlando['date'])
wx_orlando['source'] = 'open-meteo'

# new city column
wx['city'] = 'Charlottesville'
wx_orlando['city'] = 'Orlando'

# Combine both into one long-format DataFrame
wx_combined = pd.concat([wx, wx_orlando], ignore_index=True)
wx_combined

,date,temp_max,precip_mm,source,city
0,2026-09-05,29.4,2.5,open-meteo,Charlottesville
1,2026-10-10,19.2,13.1,open-meteo,Charlottesville
2,2026-09-05,32.1,3.9,open-meteo,Orlando
3,2026-10-10,30.7,9.9,open-meteo,Orlando
